[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/benchmarks/noisy_robustness.ipynb)

# MISDA — robustness to observation noise

This notebook is a presentation front end for the reproducible observation-noise sweep. Within each replicate, the clean sample and standardized noise realization are reused across all `sigma` values, so only noise intensity changes along a curve.


In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
target = f"{repo_root}[benchmarks]" if repo_root is not None else "misda[benchmarks] @ git+https://github.com/monacofj/misda.git@main"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from misda.benchmarks.validation import (
    NOISE_REPLICATE_SEEDS, NOISE_SIGMAS, NOISY_ROBUSTNESS_PROBLEM_IDS, run_noisy_robustness
)

N = 300
MISDA_SEED = 123
PROBLEM_IDS = NOISY_ROBUSTNESS_PROBLEM_IDS
SIGMAS = NOISE_SIGMAS
REPLICATE_SEEDS = NOISE_REPLICATE_SEEDS


## Controlled sweep design

The sigma grid samples the degradation curve around the fixed `0.10` reference condition used by `controlled_noisy.ipynb`; it does **not** define a pass/fail threshold.


In [ ]:
robustness_artifact = run_noisy_robustness(
    n=N, misda_seed=MISDA_SEED, problem_ids=PROBLEM_IDS, sigmas=SIGMAS, replicate_seeds=REPLICATE_SEEDS
)
robustness = pd.DataFrame(robustness_artifact["records"])
robustness_summary = pd.DataFrame(robustness_artifact["summary"])
robustness_summary


## Reading the curves

`pareto_observation_jaccard` isolates observation effects (`P_Y` vs `P_Z`), `pareto_reduction_jaccard` isolates reduction effects (`P_R` vs `P_Y`), and `pareto_end_to_end_jaccard` measures the combined effect (`P_R` vs `P_Z`). The continuous Pareto-stability diagnostics are descriptive; no fixed pass/fail cutoff is introduced.


In [ ]:
for metric in (
    "latent_recovery", "structural_recovery", "selected_unit_recovery",
    "pareto_observation_jaccard", "pareto_reduction_jaccard", "pareto_end_to_end_jaccard",
    "pareto_observed_fraction", "pareto_additive_epsilon",
    "pareto_dominance_margin_median", "supported_rate", "transitive_chaining_rate",
):
    pivot = robustness_summary.pivot(index="sigma", columns="problem_id", values=metric)
    ax = pivot.plot(marker="o", title=metric.replace("_", " ").title())
    ax.set_xlabel("sigma")
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.25)
    plt.show()
